# RAG Vector DB Setup — Unified PDF + KCC Index (Milestone 3)

Builds the production retrieval stack specified in the Milestone-3 report (Sections 9-10):
one Qdrant collection holding **both corpora** under a unified payload schema, embedded with
the MuRIL-based sentence transformer, exposed to the LLM as a single tool.

**Inputs (produced by the two preprocessing notebooks — download cells at their end):**
- `pdf_chunks_final.jsonl` — 7,136 structure-aware PDF chunks (from `04_pdfs_rag_eda.ipynb`)
- `kcc_chunks_rag.jsonl` (or `.jsonl.gz`) — 716,287 KCC Q&A chunks (from `04_kcc_preprocessing.ipynb`)

**Pipeline:**
1. **Unified ingestion** — both corpora normalized into one schema (`source_type` = `pdf` | `kcc`):
   per-chunk language re-detection (KCC's M2 tag covered only the query half), shared
   district/crop canonicalization, deterministic `chunk_id` for both corpora, token-budget
   enforcement (oversize KCC Hindi chunks re-split at sentence/danda boundaries)
2. **Embedder** — `Yunika/muril-base-sentence-transformer` (MuRIL base + mean pooling, 768-dim,
   512-token limit) with a load-time verification probe: architecture check + anisotropy
   self-test (raw `google/muril-base-cased` collapses — every pair scores ~0.99 — and must
   never silently ship)
3. **Vector DB** — Qdrant (local, on-disk), HNSW `m=16, ef_construct=128`, cosine on
   L2-normalized vectors, payload keyword indexes for filtered retrieval
4. **LLM tool** — `search_agri_knowledge(...)`: JSON-in/JSON-out, never raises, per-source
   weighted fusion (policy intent -> PDF-heavy, field-practice -> KCC-heavy), Milestone-1
   relevance tiers (grounded / fallback / abstain), page-level PDF citations
5. **Evaluation** — farmer-query battery (EN / Devanagari / Hinglish, filters, PDF-vs-KCC
   routing, off-domain abstention) with per-query latency

> **Colab free tier (T4):** embedding is the GPU stage. Full KCC (716k chunks) embeds in
> roughly 20-40 min; set `KCC_MAX_CHUNKS` in the config cell for a faster stratified subset.
> The Qdrant store is written under `RAG_DB_DIR` — copy it to Drive before the runtime dies.

In [ ]:
# --- Get the zipped data folder from Drive into Colab ---
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile

ZIP_ON_DRIVE = "/content/drive/MyDrive/data.zip"   # <-- your zip's path in Drive
EXTRACT_TO   = "/content"                          # data/ will appear at /content/data

assert os.path.exists(ZIP_ON_DRIVE), f"not found: {ZIP_ON_DRIVE}  (check the exact name/path in Drive)"
print(f"zip size: {os.path.getsize(ZIP_ON_DRIVE)/1e6:.0f} MB")

with zipfile.ZipFile(ZIP_ON_DRIVE) as zf:
    bad = zf.testzip()            # integrity check — None means the zip is intact
    assert bad is None, f"corrupted zip, first bad file: {bad}"
    zf.extractall(EXTRACT_TO)

# show what landed + verify the KCC chunks are complete
for root, _, files in os.walk("/content/data/final"):
    for fn in files:
        p = os.path.join(root, fn)
        print(f"{os.path.getsize(p)/1e6:8.1f} MB  {p}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
zip size: 400 MB
   463.3 MB  /content/data/final/kcc/kcc_chunks_rag.jsonl
     0.0 MB  /content/data/final/kcc/metadata_schema.json
    10.0 MB  /content/data/final/pdfs/pdf_chunks_final.jsonl


In [ ]:
# Dependencies. torch ships with Colab; sentencepiece backs the MuRIL tokenizer.
!pip install qdrant-client sentence-transformers transformers sentencepiece tqdm --break-system-packages -q

In [ ]:
import os

In [1]:
# ---------------------------------------------------------------------------
# Configuration — every tunable in one place (values per Milestone-3 report §9)
# ---------------------------------------------------------------------------
import os

# --- Input chunk artifacts (upload here or mount Drive and point at them) ---
# from google.colab import drive; drive.mount('/content/drive')
def _first_existing(candidates):
    for p in candidates:
        if os.path.exists(p) or os.path.exists(p + ".gz"):
            return p
    return candidates[-1]

# Repo layout first (running inside the cloned repo, chunks produced in place),
# /content fallback (Colab: upload the downloaded artifacts there).
PDF_CHUNKS_PATH = _first_existing(["../data/final/pdfs/pdf_chunks_final.jsonl",
                                   "/content/data/final/pdfs/pdf_chunks_final.jsonl"])
KCC_CHUNKS_PATH = _first_existing(["../data/final/kcc/kcc_chunks_rag.jsonl",
                                   "/content/data/final/kcc/kcc_chunks_rag.jsonl"])   # .gz auto-detected

# --- Output ---
RAG_DB_DIR      = "/content/rag_outputs" if os.path.isdir("/content") else "../data/final/rag_index"
QDRANT_PATH     = os.path.join(RAG_DB_DIR, "qdrant_db")
COLLECTION_NAME = "agri_knowledge"                      # one collection, both corpora
os.makedirs(RAG_DB_DIR, exist_ok=True)

# --- Embedder (M3 report §9.2; MuRIL base fine-tuned for sentence embeddings) ---
EMBED_MODEL_NAME = "Yunika/muril-base-sentence-transformer"
MODEL_MAX_TOKENS = 512      # MuRIL truncates silently beyond this
EMBED_BATCH_SIZE = 256      # T4-safe; halve if you hit CUDA OOM
CHUNK_TOKEN_BUDGET = 400    # re-split target for oversize KCC chunks (matches PDF chunker)

# --- KCC scale control (Colab free tier) ---
# None = index all 716k chunks (~20-40 min embedding on T4, ~2.5 GB RAM in Qdrant).
# An int (e.g. 100_000) takes a crop-stratified sample for faster iteration runs.
KCC_MAX_CHUNKS = 3000

# --- Retrieval (M3 report §9.4-9.6; M1 §10 tier thresholds) ---
TOP_K_DEFAULT  = 5
TIER_GROUNDED  = 0.85       # >= : cite-and-answer
TIER_FALLBACK  = 0.65       # >= : answer + mandatory "verify with local KVK" disclaimer
                            # <  : abstain / out-of-scope
# NOTE: M1 calibrated these thresholds before the embedder was finalized. The
# evaluation cell prints the observed score distribution — recalibrate from it.

# Per-source fusion weights by query intent (M2 §7.2 design, validated in eval below):
FUSION_WEIGHTS = {
    "policy":         {"pdf": 2.0, "kcc": 0.5},   # scheme/eligibility/subsidy questions
    "field_practice": {"pdf": 0.5, "kcc": 2.0},   # how-to / dosage / cultivation questions
    "general":        {"pdf": 1.0, "kcc": 1.0},
}

print(f"PDF chunks : {PDF_CHUNKS_PATH}  exists={os.path.exists(PDF_CHUNKS_PATH)}")
print(f"KCC chunks : {KCC_CHUNKS_PATH}  exists={os.path.exists(KCC_CHUNKS_PATH) or os.path.exists(KCC_CHUNKS_PATH + '.gz')}")
print(f"Vector DB  : {QDRANT_PATH}")

PDF chunks : /content/data/final/pdfs/pdf_chunks_final.jsonl  exists=True
KCC chunks : /content/data/final/kcc/kcc_chunks_rag.jsonl  exists=True
Vector DB  : /content/rag_outputs/qdrant_db


## 1. Unified Ingestion — PDF + KCC Coexistence

Both corpora are normalized into **one payload schema** (M3 report Appendix-B design). The three
measured schema conflicts between the corpora are resolved here, at ingestion:

| Conflict | Resolution |
|---|---|
| KCC `language` described only the *query* half (`english` for 98.8%-Devanagari answers) | `language` re-detected **per chunk** over the full text, same detector for both corpora |
| KCC districts are raw uppercase with typos (`KANPUR CITY`, `MAHARAHGANJ`); PDF uses canonical post-bifurcation names | one shared canonicalization map at ingestion; raw value preserved in `district_raw` |
| KCC chunks have no deterministic identity | `chunk_id = uuid5(content-hash : chunk_number)` for KCC; PDF chunks already carry `uuid5(sha256 : index)` |

Additionally: KCC chunks over the 512-token embedder budget (the M2 chunker was character-based
and its sentence splitter missed the Devanagari danda) are **re-split here** at sentence/danda
boundaries to the same 400-token budget the PDF chunker uses. The 98.5% single-chunk majority
passes through with field mapping only.

**Unified fields (every chunk):** `chunk_id, source_type, source, text, language, year, crop,
district, chunk_index, n_chunks_in_doc`.
**PDF-only:** `filename, doc_category, heading_hierarchy, page_start, page_end, has_table,
extraction_method, source_pdf_sha256`. **KCC-only:** `season, query_type, category, month,
district_raw`. A filter on a per-corpus field implicitly restricts to that corpus.

In [2]:
import gzip
import hashlib
import json
import re
import uuid
from collections import Counter, defaultdict

from tqdm.auto import tqdm

# ---------------------------------------------------------------------------
# Shared normalization helpers (used by BOTH corpora — this is the coexistence
# contract: same language detector, same district/crop canonical vocabulary)
# ---------------------------------------------------------------------------

def detect_language(text, sample_chars=1000):
    """Per-chunk script-ratio language tag: en / hi / mixed.
    Same thresholds as the PDF EDA notebook, so tags stay comparable."""
    s = text[:sample_chars]
    dev = len(re.findall(r"[\u0900-\u097F]", s))
    lat = len(re.findall(r"[a-zA-Z]", s))
    total = max(dev + lat, 1)
    if dev / total > 0.15 and lat / total > 0.15:
        return "mixed"
    return "hi" if dev / total > 0.3 else "en"

# District renames/typos -> canonical post-bifurcation names (extends the
# VALIDATED_ALIASES map from 04_pdfs_rag_eda.ipynb; KCC adds raw-source typos).
DISTRICT_CANON = {
    "allahabad": "prayagraj", "faizabad": "ayodhya",
    "prabuddh nagar": "shamli", "prabudh nagar": "shamli",
    "bhim nagar": "sambhal", "panchsheel nagar": "hapur",
    "jyotiba phule nagar": "amroha", "jyotibaphule nagar": "amroha",
    "kanshi ram nagar": "kasganj", "kanshiram nagar": "kasganj",
    "chhatrapati shahuji maharaj nagar": "amethi",
    "mahamaya nagar": "hathras", "ramabai nagar": "kanpur dehat",
    "banaras": "varanasi", "kashi": "varanasi",
    # raw KCC source typos observed in the corpus
    "kanpur city": "kanpur nagar", "maharahganj": "maharajganj",
    "sant ravidas nagar": "bhadohi",
}

def canon_district(raw):
    """Lowercase, collapse spaces, apply the shared rename/typo map."""
    if not raw or str(raw).lower() in ("unknown", "nan", "none", ""):
        return None
    d = re.sub(r"\s+", " ", str(raw).strip().lower())
    return DISTRICT_CANON.get(d, d)

CROP_CANON = {  # Hindi/Hinglish variants -> canonical English (matches Yield vocab)
    "paddy": "rice", "dhan": "rice", "chawal": "rice",
    "gehun": "wheat", "gehu": "wheat", "kanak": "wheat",
    "makka": "maize", "makai": "maize", "bhutta": "maize",
    "chana": "gram", "til": "sesame", "sarson": "mustard", "urd": "urad",
}

def canon_crop(raw):
    if not raw or str(raw).lower() in ("unknown", "nan", "none", ""):
        return None
    c = re.sub(r"\s+", " ", str(raw).strip().lower())
    return CROP_CANON.get(c, c)

def read_jsonl(path):
    """Stream a .jsonl or .jsonl.gz line-by-line."""
    if not path.endswith(".gz") and not os.path.exists(path) and os.path.exists(path + ".gz"):
        path = path + ".gz"
    opener = gzip.open if path.endswith(".gz") else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

# ---------------------------------------------------------------------------
# Load PDF chunks -> unified schema (metadata already rich; mostly a re-map)
# ---------------------------------------------------------------------------
unified = []

for c in tqdm(read_jsonl(PDF_CHUNKS_PATH), desc="PDF chunks"):
    m = c["metadata"]
    filename = m.get("filename", "")
    # ACP files are district contingency plans named after the district
    district = canon_district(os.path.splitext(filename)[0]) if m.get("source") == "up_acp" else None
    unified.append({
        # shared
        "chunk_id": m["chunk_id"],
        "source_type": "pdf",
        "source": m.get("source", ""),
        "text": c["text"],
        "language": m.get("detected_language") or detect_language(c["text"]),
        "year": int(m["detected_year"]) if m.get("detected_year") else None,
        "crop": None,                      # document-level crop tagging: future work
        "district": district,
        "chunk_index": m.get("chunk_index", 0),
        "n_chunks_in_doc": m.get("n_chunks_in_doc", 1),
        # pdf-only
        "filename": filename,
        "doc_category": m.get("doc_category", ""),
        "heading_hierarchy": m.get("heading_hierarchy", ""),
        "page_start": m.get("page_start"),
        "page_end": m.get("page_end"),
        "has_table": bool(m.get("has_table", False)),
        "extraction_method": m.get("extraction_method", ""),
        "source_pdf_sha256": m.get("source_pdf_sha256", ""),
    })

n_pdf = len(unified)
print(f"PDF chunks loaded: {n_pdf:,}")
print(f"  with page provenance: {sum(1 for u in unified if u['page_start'] is not None):,}")
print(f"  with district (ACP):  {sum(1 for u in unified if u['district']):,}")
print(f"  language mix: {Counter(u['language'] for u in unified)}")

PDF chunks: 0it [00:00, ?it/s]

PDF chunks loaded: 7,136
  with page provenance: 7,117
  with district (ACP):  4,818
  language mix: Counter({'en': 7052, 'mixed': 84})


In [3]:
# ---------------------------------------------------------------------------
# Load + normalize KCC chunks -> unified schema
#   - deterministic chunk_id (uuid5 over content hash)
#   - per-chunk language re-detection (full Q+A text, not just the query)
#   - shared district/crop canonicalization
#   - token-budget enforcement: oversize chunks re-split at sentence/danda
# ---------------------------------------------------------------------------
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_NAME)

def n_tokens(text):
    return len(tokenizer.encode(text, add_special_tokens=False))

SENT_SPLIT = re.compile(r"(?<=[.!?।])\s+")   # । = Devanagari danda

def split_to_budget(text, budget=CHUNK_TOKEN_BUDGET):
    """Sentence-accumulating re-split for the oversize tail (danda-aware).
    Only called for chunks over the embedder budget — ~1.5% of the corpus."""
    sents = SENT_SPLIT.split(text)
    parts, cur, cur_tok = [], [], 0
    for s in sents:
        t = n_tokens(s)
        if cur and cur_tok + t > budget:
            parts.append(" ".join(cur))
            cur, cur_tok = [], 0
        cur.append(s)
        cur_tok += t
    if cur:
        parts.append(" ".join(cur))
    return parts or [text]

def kcc_chunk_id(text, meta, chunk_no, seq):
    # `seq` = source-record index in the file. KCC dedup removed exact
    # (query, answer, crop) duplicates, but near-duplicate call records sharing
    # the same leading text + crop/district/time survive; hashing full text plus
    # the record index guarantees a unique id for every chunk while staying
    # deterministic for a given input file (so re-ingestion is idempotent).
    basis = "|".join([text, str(meta.get("crop")), str(meta.get("district")),
                      str(meta.get("year")), str(meta.get("month")),
                      str(seq), str(chunk_no)])
    h = hashlib.sha1(basis.encode("utf-8")).hexdigest()
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f"kcc:{h}"))

kcc_rows, n_resplit, n_seen = [], 0, 0
budget_hard = MODEL_MAX_TOKENS - 2

for c in tqdm(read_jsonl(KCC_CHUNKS_PATH), desc="KCC chunks"):
    n_seen += 1
    m = c.get("metadata", {})
    text = c.get("text", "")
    if len(text.strip()) < 10:
        continue

    # cheap length pre-filter: only tokenize candidates that could exceed budget
    pieces = [text]
    if len(text) > 900:                          # ~ >250 tokens; safe lower bound
        if n_tokens(text) > budget_hard:
            pieces = split_to_budget(text)
            n_resplit += 1

    year = m.get("year")
    base = {
        "source_type": "kcc",
        "source": "kcc_qa",
        "language": detect_language(text),
        "year": int(year) if year else None,
        "crop": canon_crop(m.get("crop")),
        "district": canon_district(m.get("district")),
        "district_raw": m.get("district"),
        "season": m.get("season") if m.get("season") not in (None, "unknown") else None,
        "query_type": m.get("query_type"),
        "category": m.get("category"),
        "month": int(m["month"]) if m.get("month") else None,
    }
    for i, piece in enumerate(pieces):
        row = dict(base)
        row["text"] = piece
        row["chunk_index"] = i if len(pieces) > 1 else int(c.get("chunk_number", 1)) - 1
        row["n_chunks_in_doc"] = len(pieces) if len(pieces) > 1 else int(c.get("total_chunks", 1))
        row["chunk_id"] = kcc_chunk_id(piece, m, row["chunk_index"], n_seen)
        kcc_rows.append(row)

print(f"KCC chunks read: {n_seen:,} -> normalized rows: {len(kcc_rows):,} "
      f"(oversize re-split: {n_resplit:,})")

# --- Optional crop-stratified subsample for fast Colab iteration -------------
if KCC_MAX_CHUNKS and len(kcc_rows) > KCC_MAX_CHUNKS:
    import random
    random.seed(42)
    by_crop = defaultdict(list)
    for r in kcc_rows:
        by_crop[r["crop"] or "_none"].append(r)
    frac = KCC_MAX_CHUNKS / len(kcc_rows)
    sampled = []
    for crop, rows in by_crop.items():
        k = max(1, round(len(rows) * frac))
        sampled.extend(random.sample(rows, min(k, len(rows))))
    kcc_rows = sampled[:KCC_MAX_CHUNKS]
    print(f"Stratified subsample applied: {len(kcc_rows):,} chunks "
          f"across {len(by_crop)} crops (KCC_MAX_CHUNKS={KCC_MAX_CHUNKS:,})")

unified.extend(kcc_rows)
print(f"\nUnified corpus: {len(unified):,} chunks "
      f"(pdf={n_pdf:,}, kcc={len(unified) - n_pdf:,})")

KCC chunks: 0it [00:00, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (837 > 512). Running this sequence through the model will result in indexing errors


KCC chunks read: 716,287 -> normalized rows: 716,298 (oversize re-split: 12)
Stratified subsample applied: 3,000 chunks across 281 crops (KCC_MAX_CHUNKS=3,000)

Unified corpus: 10,136 chunks (pdf=7,136, kcc=3,000)


In [4]:
# ---------------------------------------------------------------------------
# Corpus summary — the coexistence picture at a glance
# ---------------------------------------------------------------------------
import pandas as pd

udf = pd.DataFrame(unified)
print(f"Total chunks: {len(udf):,}\n")
print("By source_type:")
print(udf["source_type"].value_counts().to_string())
print("\nLanguage (re-detected per chunk — note KCC is mostly `mixed`/`hi`,")
print("not the `english` the M2 query-only tag suggested):")
print(udf.groupby("source_type")["language"].value_counts().to_string())
print("\nTop districts after shared canonicalization:")
print(udf["district"].value_counts().head(8).to_string())
print("\nTop crops after canonicalization (KCC):")
print(udf[udf.source_type == "kcc"]["crop"].value_counts().head(8).to_string())

dup = len(udf) - udf["chunk_id"].nunique()
assert dup == 0, f"{dup} duplicate chunk_ids — deterministic identity broken"
print(f"\nchunk_id uniqueness: OK ({udf['chunk_id'].nunique():,} unique)")
del udf

Total chunks: 10,136

By source_type:
source_type
pdf    7136
kcc    3000

Language (re-detected per chunk — note KCC is mostly `mixed`/`hi`,
not the `english` the M2 query-only tag suggested):
source_type  language
kcc          mixed       2919
             en            47
             hi            34
pdf          en          7052
             mixed         84

Top districts after shared canonicalization:
district
bareilly        191
badaun          181
bulandshahar    171
ghazipur        168
shahjahanpur    160
sitapur         158
gonda           144
gorakhpur       142

Top crops after canonicalization (KCC):
crop
paddy (dhan)              470
wheat                     401
sugarcane (noble cane)    246
potato                    171
mango                     154
mustard                   152
chillies                   76
pea (vegetable)            69

chunk_id uniqueness: OK (10,136 unique)


## 2. Embedding Model — MuRIL Sentence Transformer (verified load)

`Yunika/muril-base-sentence-transformer`: MuRIL base (BERT, 17 Indian languages incl.
transliterated Hindi) with TripletLoss sentence fine-tuning, mean pooling, 768-dim,
512-token limit — loaded exactly per its model card via `SentenceTransformer(...)`.

The load is **verified, not trusted**: raw `google/muril-base-cased` is an MLM checkpoint whose
mean-pooled vectors collapse anisotropically (every pair scores ~0.99 — agriculture becomes
indistinguishable from chess; measured during prototyping). The probe below fails the notebook
rather than let a collapsed space silently reach the index. It also reports an EN-HI
cross-lingual margin: if that warns, swap `EMBED_MODEL_NAME` to
`l3cube-pune/indic-sentence-similarity-sbert` (MuRIL-family, NLI+STS-tuned on 10 Indic
languages) — a one-line change.

In [5]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else " — embedding will be slow; enable the GPU runtime"))

embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=device)
EMBED_DIM = embed_model.get_sentence_embedding_dimension()
print(f"Model: {EMBED_MODEL_NAME} | dim={EMBED_DIM}")

# --- Verification probe (per model card: Transformer(512) + mean Pooling) ----
print("\nModule stack (must be Transformer + Pooling from the repo config):")
for name, module in embed_model.named_children():
    print(f"  ({name}): {module}")

assert embed_model.max_seq_length == MODEL_MAX_TOKENS == 512, \
    f"max_seq_length mismatch: model={embed_model.max_seq_length}, config={MODEL_MAX_TOKENS}"
# Pooling check, robust across sentence-transformers versions:
#   v5 exposes a `pooling_mode` STRING ("mean"); older builds a
#   `pooling_mode_mean_tokens` BOOLEAN; get_pooling_mode_str() is gone in 3.x+.
_pool = embed_model[1]
try:
    _pcfg = _pool.get_config_dict()
except Exception:
    _pcfg = {}
_is_mean = (str(_pcfg.get("pooling_mode", "")).lower() == "mean"
            or bool(_pcfg.get("pooling_mode_mean_tokens", False))
            or getattr(_pool, "pooling_mode", None) == "mean"
            or bool(getattr(_pool, "pooling_mode_mean_tokens", False)))
assert _is_mean, f"expected mean pooling per model card; pooling config = {_pcfg}"
print("pooling: mean  (verified)")

# Anisotropy self-test: a collapsed space scores ANY pair ~0.99 (raw-MuRIL
# failure signature). Related must beat unrelated by a clear margin.
_probe = embed_model.encode(
    ["blast disease treatment in rice crop",
     "धान की फसल में ब्लास्ट रोग का उपचार",     # same meaning, Hindi
     "how to win a game of chess"],             # unrelated
    normalize_embeddings=True,
)
_rel, _unrel = float(_probe[0] @ _probe[1]), float(_probe[0] @ _probe[2])
print(f"\nEmbedding sanity — related(EN~HI): {_rel:.3f}, unrelated: {_unrel:.3f}, "
      f"margin: {_rel - _unrel:+.3f}")
if _unrel > 0.95:
    raise RuntimeError(
        "Anisotropic collapse detected — this checkpoint does not produce usable "
        "sentence embeddings. Do NOT build the index. Swap EMBED_MODEL_NAME "
        "(e.g. l3cube-pune/indic-sentence-similarity-sbert) and re-run.")
if _rel <= _unrel:
    print("[WARN] EN~HI pair does not beat the unrelated pair — Hindi retrieval "
          "will be weak (English-only fine-tune limitation). Consider the L3Cube fallback.")

Device: cuda (Tesla T4)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_5327/4176202427.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMBED_DIM = embed_model.get_sentence_embedding_dimension()


Model: Yunika/muril-base-sentence-transformer | dim=768

Module stack (must be Transformer + Pooling from the repo config):
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
pooling: mean  (verified)

Embedding sanity — related(EN~HI): 0.927, unrelated: 0.053, margin: +0.874


In [6]:
# ---------------------------------------------------------------------------
# Embedding is STREAMED shard-by-shard during indexing (next cell), so the full
# embedding array is NEVER held in RAM at once. Holding the whole array WHILE
# Qdrant builds the HNSW index is what OOM-crashed the one-shot version. Here we
# only set the shard size; the model is already loaded and sanity-checked above.
# The corpus size is whatever the config cell selected (DEV subset or full) —
# every chunk in `unified` is indexed; nothing is dropped at this stage.
# ---------------------------------------------------------------------------
SHARD_SIZE = 50_000   # ~150 MB of float32 vectors per shard; lower this if RAM is still tight
_n = (len(unified) + SHARD_SIZE - 1) // SHARD_SIZE
print(f"Corpus: {len(unified):,} chunks -> {_n} shards of up to {SHARD_SIZE:,}")
print("Embedding runs shard-by-shard in the build cell below (peak RAM = one shard).")


Corpus: 10,136 chunks -> 1 shards of up to 50,000
Embedding runs shard-by-shard in the build cell below (peak RAM = one shard).


## 3. Qdrant — Sharded Build, HNSW + Cosine, Payload Indexes

One `agri_knowledge` collection for both corpora (M3 report Sec 5.5 / 9.3-9.4): HNSW
`m=16, ef_construct=128`, cosine on L2-normalized 768-dim vectors, on-disk vectors,
payload keyword indexes for filtered retrieval.

**Full corpus, memory-safe build.** Embedding + upsert run in **shards of ~50k** — each
shard is embedded, upserted, then freed, so peak RAM is one shard (~150 MB) instead of
the whole 2.2 GB array. HNSW indexing is **deferred** (`indexing_threshold=0` during
upload, flipped on at the end) so it builds once rather than thrashing during insert.
A `shard_progress.json` checkpoint makes the build **resumable** after an interruption.
No chunks are dropped — all 723k are indexed.

> Point IDs are the deterministic `chunk_id`s, so re-running is idempotent and resumable.
> `indexing_threshold` must be POSITIVE to build HNSW; `0` (used only during upload here)
> disables it.


In [7]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, HnswConfigDiff, OptimizersConfigDiff, PointStruct, VectorParams,
)
import gc
import json
import time

import numpy as np
from tqdm.auto import tqdm

qdrant_client = QdrantClient(path=QDRANT_PATH)   # local/embedded: EXACT search
# (local mode ignores HNSW + payload indexes and does exact brute-force NN --
#  highest recall, just slower per query; the HNSW config is the server-mode path).
PROGRESS_PATH = os.path.join(RAG_DB_DIR, "shard_progress.json")
n_shards = (len(unified) + SHARD_SIZE - 1) // SHARD_SIZE

# --- Resume an interrupted build, or start clean ---------------------------
done, fresh = set(), True
if qdrant_client.collection_exists(COLLECTION_NAME) and os.path.exists(PROGRESS_PATH):
    done = set(json.load(open(PROGRESS_PATH)))
    fresh = False
    print(f"Resuming build: {len(done)}/{n_shards} shards already uploaded (skipping those)")

if fresh:
    if qdrant_client.collection_exists(COLLECTION_NAME):
        qdrant_client.delete_collection(COLLECTION_NAME)
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE, on_disk=True),
        hnsw_config=HnswConfigDiff(m=16, ef_construct=128),
        # Indexing OFF during bulk upload; built once at the end. This avoids the
        # incremental-index RAM/CPU spike that OOM-crashed the single-shot build.
        optimizers_config=OptimizersConfigDiff(indexing_threshold=0),
    )
    for field in ["source_type", "source", "language", "doc_category",
                  "crop", "district", "season", "query_type"]:
        qdrant_client.create_payload_index(COLLECTION_NAME, field_name=field, field_schema="keyword")
    qdrant_client.create_payload_index(COLLECTION_NAME, field_name="year", field_schema="integer")
    qdrant_client.create_payload_index(COLLECTION_NAME, field_name="has_table", field_schema="bool")
    json.dump([], open(PROGRESS_PATH, "w"))
    print(f"Fresh collection '{COLLECTION_NAME}' created (HNSW indexing deferred)")

# --- Sharded: embed -> upsert -> free. Peak RAM = one shard (~150 MB) ------
UPSERT_BATCH = 256
done_chunks = sum(min((k + 1) * SHARD_SIZE, len(unified)) - k * SHARD_SIZE for k in done)

# Outer bar = overall progress across the whole corpus (with ETA); the embedder's
# own per-batch bar (show_progress_bar=True) shows live progress WITHIN each shard,
# so the long embedding stretches no longer look frozen.
pbar = tqdm(total=len(unified), initial=done_chunks, desc="Indexing corpus", unit="chunk")
for s in range(n_shards):
    if s in done:
        continue
    lo, hi = s * SHARD_SIZE, min((s + 1) * SHARD_SIZE, len(unified))
    shard = unified[lo:hi]
    pbar.set_postfix_str(f"shard {s + 1}/{n_shards} (embedding...)")
    vecs = embed_model.encode(
        [r["text"] for r in shard], batch_size=EMBED_BATCH_SIZE,
        show_progress_bar=True, normalize_embeddings=True, convert_to_numpy=True,
    ).astype(np.float32)
    pbar.set_postfix_str(f"shard {s + 1}/{n_shards} (upserting...)")
    for b in range(0, len(shard), UPSERT_BATCH):
        pts = [
            PointStruct(id=r["chunk_id"], vector=vecs[b + j].tolist(),
                        payload={k: v for k, v in r.items() if v is not None})
            for j, r in enumerate(shard[b:b + UPSERT_BATCH])
        ]
        qdrant_client.upsert(collection_name=COLLECTION_NAME, points=pts)
    done.add(s)
    json.dump(sorted(done), open(PROGRESS_PATH, "w"))   # checkpoint after each shard
    pbar.update(hi - lo)
    del vecs, shard
    gc.collect()
pbar.close()

# --- All points in: enable indexing -> HNSW builds once --------------------
print()
print("All shards uploaded. Enabling HNSW indexing (builds in background)...")
qdrant_client.update_collection(
    collection_name=COLLECTION_NAME,
    optimizers_config=OptimizersConfigDiff(indexing_threshold=100),
)
# Search already works before this finishes (brute-force on unindexed segments);
# poll until the optimizer reports GREEN so queries are fast.
deadline = time.time() + 1800
while time.time() < deadline:
    info = qdrant_client.get_collection(COLLECTION_NAME)
    if str(info.status).split(".")[-1].lower() == "green":
        break
    time.sleep(15)

info = qdrant_client.get_collection(COLLECTION_NAME)
print()
print(f"Collection '{COLLECTION_NAME}' ready")
print(f"  points:        {info.points_count:,}")
print(f"  indexed vecs:  {info.indexed_vectors_count:,} / {info.points_count:,}")
print(f"  status:        {info.status}")
print(f"  storage:       {QDRANT_PATH}")
print("  -> copy this folder (+ shard_progress.json) to Drive to persist across sessions!")


Fresh collection 'agri_knowledge' created (HNSW indexing deferred)


/tmp/ipykernel_5327/1296649931.py:38: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  qdrant_client.create_payload_index(COLLECTION_NAME, field_name=field, field_schema="keyword")


Indexing corpus:   0%|          | 0/10136 [00:00<?, ?chunk/s]

Batches:   0%|          | 0/40 [00:00<?, ?it/s]


All shards uploaded. Enabling HNSW indexing (builds in background)...

Collection 'agri_knowledge' ready
  points:        10,136
  indexed vecs:  0 / 10,136
  status:        green
  storage:       /content/rag_outputs/qdrant_db
  -> copy this folder (+ shard_progress.json) to Drive to persist across sessions!


## 4. LLM Tool — `search_agri_knowledge`

The single retrieval entrypoint the agentic LLM registers (M3 report §10.7, Appendix D):
JSON-in / JSON-out, **never raises** (errors return `tier: "error"`), every hit cited
(PDF -> file + pages + section; KCC -> record + district + year).

**Coexistence at query time — weighted per-source fusion, not one flat search.** The KCC
corpus outnumbers PDF ~100:1; a flat top-k would drown scheme/policy content. The tool runs
one filtered sub-query per corpus and fuses with intent weights (`policy` -> PDF 2.0/KCC 0.5,
`field_practice` -> KCC 2.0/PDF 0.5, `general` -> 1.0/1.0). Callers can also pin
`source_type` explicitly. The Milestone-1 relevance tier (grounded / fallback / abstain)
is decided on the best **raw** cosine score, never the fused one.

In [8]:
# Filter primitives for the tool (self-contained -- the build cell no longer
# exports these into the global scope).
from qdrant_client.models import Filter, FieldCondition, MatchValue, Range

RAG_TOOL_SPEC = {
    "name": "search_agri_knowledge",
    "description": (
        "Search the UP agricultural knowledge base: government scheme guidelines, "
        "pest/disease advisories and district contingency plans (PDF corpus) plus "
        "Kisan Call Centre farmer Q&A with expert answers (KCC corpus). Returns "
        "cited chunks with a relevance tier. Query may be English or Hindi."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "query":       {"type": "string", "description": "The farmer's question, English or Hindi"},
            "top_k":       {"type": "integer", "default": 5, "minimum": 1, "maximum": 20},
            "intent":      {"type": "string", "enum": ["policy", "field_practice", "general"],
                            "description": "Weights the PDF-vs-KCC fusion; default general"},
            "source_type": {"type": "string", "enum": ["pdf", "kcc"],
                            "description": "Pin one corpus; omit for weighted search over both"},
            "doc_category": {"type": "string", "enum": ["scheme_eligibility", "crop_advisory",
                                                        "contingency_plan", "policy_guideline"],
                             "description": "PDF corpus only"},
            "query_type":  {"type": "string", "description": "KCC only, e.g. 'Plant Protection'"},
            "crop":        {"type": "string", "description": "Canonical crop name (rice, wheat, ...)"},
            "district":    {"type": "string", "description": "Canonical UP district name"},
            "season":      {"type": "string", "enum": ["Rabi", "Kharif", "Zaid"], "description": "KCC only"},
            "language":    {"type": "string", "enum": ["en", "hi", "mixed"]},
            "year_from":   {"type": "integer", "description": "Only content from this year onward"},
            "only_tables": {"type": "boolean", "description": "PDF dosage/scheme tables only"},
        },
        "required": ["query"],
    },
}


def _citation(p):
    if p.get("source_type") == "pdf":
        return {"corpus": "pdf", "file": p.get("filename"),
                "pages": [p.get("page_start"), p.get("page_end")],
                "section": p.get("heading_hierarchy") or None,
                "doc_category": p.get("doc_category"), "year": p.get("year")}
    return {"corpus": "kcc", "record": "KCC Q&A", "crop": p.get("crop"),
            "district": p.get("district"), "season": p.get("season"),
            "query_type": p.get("query_type"), "year": p.get("year")}


def search_agri_knowledge(query, top_k=TOP_K_DEFAULT, intent="general", source_type=None,
                          doc_category=None, query_type=None, crop=None, district=None,
                          season=None, language=None, year_from=None, only_tables=None):
    """LLM tool entrypoint: JSON-in/JSON-out, never raises, always cites."""
    weights = FUSION_WEIGHTS.get(intent, FUSION_WEIGHTS["general"])

    def sub_search(stype):
        must = [FieldCondition(key="source_type", match=MatchValue(value=stype))]
        if doc_category: must.append(FieldCondition(key="doc_category", match=MatchValue(value=doc_category)))
        if query_type:   must.append(FieldCondition(key="query_type", match=MatchValue(value=query_type)))
        if crop:         must.append(FieldCondition(key="crop", match=MatchValue(value=canon_crop(crop))))
        if district:     must.append(FieldCondition(key="district", match=MatchValue(value=canon_district(district))))
        if season:       must.append(FieldCondition(key="season", match=MatchValue(value=season)))
        if language:     must.append(FieldCondition(key="language", match=MatchValue(value=language)))
        if year_from:    must.append(FieldCondition(key="year", range=Range(gte=year_from)))
        if only_tables:  must.append(FieldCondition(key="has_table", match=MatchValue(value=True)))
        return qdrant_client.query_points(
            collection_name=COLLECTION_NAME,
            query=qvec,
            query_filter=Filter(must=must),
            limit=top_k,
            with_payload=True,
        ).points

    try:
        qvec = embed_model.encode(query, normalize_embeddings=True).tolist()
        sources = [source_type] if source_type else ["pdf", "kcc"]
        hits = []
        for stype in sources:
            for h in sub_search(stype):
                hits.append({
                    "raw_score": round(float(h.score), 4),
                    "fused_score": round(float(h.score) * weights.get(stype, 1.0), 4),
                    "text": h.payload.get("text", ""),
                    "source_type": stype,
                    "has_table": bool(h.payload.get("has_table", False)),
                    "chunk_id": h.payload.get("chunk_id"),
                    "citation": _citation(h.payload),
                })
        hits.sort(key=lambda x: x["fused_score"], reverse=True)
        hits = hits[:top_k]
    except Exception as e:
        return {"query": query, "tier": "error", "top_score": 0.0,
                "results": [], "error": str(e)}

    best_raw = max((h["raw_score"] for h in hits), default=0.0)  # tier on RAW cosine
    tier = ("grounded" if best_raw >= TIER_GROUNDED
            else "fallback_with_disclaimer" if best_raw >= TIER_FALLBACK
            else "abstain_out_of_scope")
    return {"query": query, "intent": intent, "tier": tier,
            "top_score": round(best_raw, 4), "results": hits}


print("Tool ready: search_agri_knowledge(...)")
print(json.dumps(RAG_TOOL_SPEC, indent=2)[:600] + " ...")

# Smoke call
_s = search_agri_knowledge("interest subvention on crop loans", top_k=3, intent="policy")
print(f"\nSmoke: tier={_s['tier']} top={_s['top_score']} "
      f"sources={[r['source_type'] for r in _s['results']]}")

Tool ready: search_agri_knowledge(...)
{
  "name": "search_agri_knowledge",
  "description": "Search the UP agricultural knowledge base: government scheme guidelines, pest/disease advisories and district contingency plans (PDF corpus) plus Kisan Call Centre farmer Q&A with expert answers (KCC corpus). Returns cited chunks with a relevance tier. Query may be English or Hindi.",
  "parameters": {
    "type": "object",
    "properties": {
      "query": {
        "type": "string",
        "description": "The farmer's question, English or Hindi"
      },
      "top_k": {
        "type": "integer",
        "default": 5,
        "minimum ...

Smoke: tier=grounded top=0.9996 sources=['pdf', 'pdf', 'pdf']


## 5. Evaluation — Farmer Queries Through the Tool

Every check below goes through `search_agri_knowledge` exactly as the LLM will call it.

- **A. Filter correctness** — every returned hit must satisfy its metadata filter
- **B. Bilingual retrieval** — the same intent in English / Devanagari / Hinglish should
  agree on what it retrieves (M1 objective O3 is code-mixed retrieval)
- **C. Corpus routing** — `policy` intent should surface PDF scheme content at the top;
  `field_practice` should surface KCC expert answers (validates the fusion weights)
- **D. Domain separation** — off-domain queries (car repair, chess, stock market) must score
  clearly below in-domain ones, or tier-based abstention cannot work; the observed
  distribution is the input for recalibrating the M1 thresholds
- **E. Latency** — per-call wall time (embed + filtered HNSW + fusion)

Output: PASS/WARN/FAIL table + `rag_eval_report.json` beside the Qdrant store.

In [9]:
report, timings = [], []

def timed_call(**kw):
    t0 = time.time()
    out = search_agri_knowledge(**kw)
    ms = (time.time() - t0) * 1000
    timings.append(ms)
    out["_ms"] = round(ms, 1)
    return out

def check(label, out, predicate):
    hits = out["results"]
    if out["tier"] == "error":
        report.append((label, "FAIL", out.get("error", "")[:80]))
    elif not hits:
        report.append((label, "WARN", "0 results — over-restrictive filter or corpus gap"))
    else:
        bad = [h for h in hits if not predicate(h)]
        report.append((label, "PASS" if not bad else "FAIL",
                       f"{len(hits)} hits, {len(bad)} violate, top={out['top_score']:.3f}, "
                       f"tier={out['tier']}, {out['_ms']:.0f}ms"))

# --- A. Filter correctness ---------------------------------------------------
check("A1 pdf-only + scheme category",
      timed_call(query="who is eligible for interest subvention on crop loans",
                 source_type="pdf", doc_category="scheme_eligibility"),
      lambda h: h["source_type"] == "pdf" and h["citation"]["doc_category"] == "scheme_eligibility")
check("A2 kcc-only + crop=rice",
      timed_call(query="fertilizer dose for paddy nursery", source_type="kcc", crop="rice"),
      lambda h: h["source_type"] == "kcc" and h["citation"]["crop"] == "rice")
check("A3 dosage tables only (pdf)",
      timed_call(query="approved fungicides and dosage for rice blast", only_tables=True),
      lambda h: h["has_table"])
check("A4 district filter (canonicalized: allahabad->prayagraj)",
      timed_call(query="crop advice for my district", district="allahabad"),
      lambda h: h["citation"].get("district") == "prayagraj")
check("A5 season filter (kcc Rabi)",
      timed_call(query="wheat sowing time", season="Rabi"),
      lambda h: h["citation"].get("season") == "Rabi")
check("A6 year_from=2020",
      timed_call(query="latest pest advisory", year_from=2020),
      lambda h: (h["citation"].get("year") or 0) >= 2020)

# --- B. Bilingual / code-mixed consistency -----------------------------------
INTENTS = [
    ("wheat irrigation", ["when should wheat be irrigated and how many times",
                          "गेहूं की सिंचाई कब और कितनी बार करें",
                          "gehu me sinchai kab karni chahiye"]),
    ("rice blast",       ["treatment for blast disease in paddy",
                          "धान में ब्लास्ट रोग का उपचार",
                          "dhan me blast rog ka ilaj"]),
]
for label, forms in INTENTS:
    got = []
    for q in forms:
        out = timed_call(query=q, top_k=5)
        got.append({(r["citation"].get("file") or r["chunk_id"]) for r in out["results"]})
    j_en_hi = len(got[0] & got[1]) / max(len(got[0] | got[1]), 1)
    hing = len(got[0] & got[2])
    report.append((f"B  bilingual [{label}]", "PASS" if j_en_hi >= 0.2 else "WARN",
                   f"EN~HI jaccard={j_en_hi:.2f}, hinglish overlap with EN={hing}/5"))

# --- C. Corpus routing via fusion weights ------------------------------------
pol = timed_call(query="pm kisan samman nidhi eligibility and benefits", intent="policy")
n_pdf_top = sum(1 for r in pol["results"][:3] if r["source_type"] == "pdf")
report.append(("C1 policy intent -> pdf-heavy top-3",
               "PASS" if n_pdf_top >= 2 else "WARN",
               f"{n_pdf_top}/3 pdf in top-3, tier={pol['tier']}"))
fld = timed_call(query="yellowing in onion nursery what spray to use", intent="field_practice")
n_kcc_top = sum(1 for r in fld["results"][:3] if r["source_type"] == "kcc")
report.append(("C2 field intent -> kcc-heavy top-3",
               "PASS" if n_kcc_top >= 2 else "WARN",
               f"{n_kcc_top}/3 kcc in top-3, tier={fld['tier']}"))

# --- D. Domain separation (abstention viability) -----------------------------
in_dom = ["recommended fertilizer schedule for wheat in up",
          "fall army worm control in maize",
          "गेहूं में पीला रतुआ की रोकथाम"]
off_dom = ["how do I repair my car engine", "best chess opening strategy",
           "शेयर बाजार में निवेश कैसे करें"]
in_s = [timed_call(query=q, top_k=1)["top_score"] for q in in_dom]
off_s = [timed_call(query=q, top_k=1)["top_score"] for q in off_dom]
margin = min(in_s) - max(off_s)
report.append(("D1 domain separation", "PASS" if margin > 0.03 else "WARN",
               f"in-domain min={min(in_s):.3f}, off-domain max={max(off_s):.3f}, margin={margin:+.3f}"))
report.append(("D2 tier calibration", "INFO",
               f"M1 thresholds {TIER_FALLBACK}/{TIER_GROUNDED} vs observed in-domain "
               f"{[round(s,3) for s in in_s]} — suggested abstain cutoff ~"
               f"{round((min(in_s)+max(off_s))/2, 3)}"))

# --- Report ------------------------------------------------------------------
print("=" * 92)
print("RAG EVALUATION — farmer queries through search_agri_knowledge")
print("=" * 92)
counts = Counter(s for _, s, _ in report)
for label, status, detail in report:
    print(f"  [{status:4}] {label:44} {detail}")
print("-" * 92)
print(f"  {counts['PASS']} pass / {counts['FAIL']} fail / {counts['WARN']} warn / "
      f"{counts['INFO']} info | latency p50={np.percentile(timings,50):.0f}ms "
      f"p95={np.percentile(timings,95):.0f}ms over {len(timings)} calls")

eval_path = os.path.join(RAG_DB_DIR, "rag_eval_report.json")
with open(eval_path, "w", encoding="utf-8") as f:
    json.dump({"checks": [dict(zip(("label","status","detail"), r)) for r in report],
               "latency_ms": {"p50": float(np.percentile(timings, 50)),
                              "p95": float(np.percentile(timings, 95))},
               "collection": COLLECTION_NAME, "model": EMBED_MODEL_NAME,
               "n_chunks": len(unified)}, f, indent=2, ensure_ascii=False)
print(f"  saved: {eval_path}")

# --- Sample tool response (what the LLM will actually receive) ---------------
print("\nSAMPLE — search_agri_knowledge('paddy blast dose', only_tables=True):")
print(json.dumps(timed_call(query="paddy blast treatment dose", top_k=2, only_tables=True),
                 indent=2, ensure_ascii=False)[:1600] + " ...")

RAG EVALUATION — farmer queries through search_agri_knowledge
  [PASS] A1 pdf-only + scheme category                5 hits, 0 violate, top=0.996, tier=grounded, 255ms
  [WARN] A2 kcc-only + crop=rice                      0 results — over-restrictive filter or corpus gap
  [PASS] A3 dosage tables only (pdf)                  5 hits, 0 violate, top=0.984, tier=grounded, 427ms
  [FAIL] A4 district filter (canonicalized: allahabad->prayagraj) 5 hits, 3 violate, top=0.993, tier=grounded, 448ms
  [PASS] A5 season filter (kcc Rabi)                  5 hits, 0 violate, top=0.945, tier=grounded, 423ms
  [PASS] A6 year_from=2020                            5 hits, 0 violate, top=0.991, tier=grounded, 417ms
  [PASS] B  bilingual [wheat irrigation]              EN~HI jaccard=0.25, hinglish overlap with EN=4/5
  [PASS] B  bilingual [rice blast]                    EN~HI jaccard=0.43, hinglish overlap with EN=0/5
  [PASS] C1 policy intent -> pdf-heavy top-3          3/3 pdf in top-3, tier=grounded
  [PA

In [10]:
# ---------------------------------------------------------------------------
# 6. Retrieval Inspector — the ACTUAL chunks behind each kind of query
# ---------------------------------------------------------------------------
# Section 5 reports PASS/WARN/FAIL but never shows WHAT came back. This cell
# prints the retrieved text itself so retrieval quality can be judged by eye,
# across the full query space the system has to handle:
#   A  policy / scheme       -> should route PDF-heavy   (FUSION_WEIGHTS)
#   B  field practice / how-to -> should route KCC-heavy (FUSION_WEIGHTS)
#   C  disease diagnosis     -> the vision-module handoff (M3 report §3)
#   D  metadata-filtered     -> filters + district canonicalization (M3 §9.5)
#   E  edge cases            -> off-domain abstention, vague, compound queries
# A/B/C are each run in English / Devanagari / Hinglish where relevant —
# code-mixed retrieval is the specific reason MuRIL was chosen (M3 §5.2).
# ---------------------------------------------------------------------------

SNIPPET_CHARS = 300      # chars of chunk text to print; set to None for the full chunk
TOP_N_SHOW    = 3        # how many hits to display per query


def _snip(text, n=SNIPPET_CHARS):
    """Collapse whitespace (KCC chunks are multi-line Q&A) and truncate."""
    t = re.sub(r"\s+", " ", str(text or "")).strip()
    if n is None or len(t) <= n:
        return t
    return t[:n].rstrip() + " ..."


def _fmt_citation(c):
    """One-line provenance; PDF and KCC carry different fields."""
    if c.get("corpus") == "pdf":
        bits = [str(c.get("file") or "?")]
        pages = c.get("pages") or [None, None]
        ps, pe = (pages + [None, None])[:2]
        if ps is not None:
            bits.append(f"p.{ps}" if pe in (None, ps) else f"pp.{ps}-{pe}")
        for k in ("doc_category", "year"):
            if c.get(k):
                bits.append(str(c[k]))
        if c.get("section"):
            bits.append(f"sec: {_snip(c['section'], 60)}")
        return " | ".join(bits)
    bits = ["KCC Q&A"]
    for k in ("crop", "district", "season", "query_type", "year"):
        if c.get(k):
            bits.append(str(c[k]))
    return " | ".join(bits)


# --- The query battery: (category, label, kwargs passed to the tool) ---------
INSPECT_QUERIES = [
    # A. Policy / scheme — fusion should push PDF to the top
    ("A. POLICY / SCHEME  (expect PDF-heavy)", "EN  · loan interest subvention",
     dict(query="who is eligible for interest subvention on crop loans", intent="policy")),
    ("A. POLICY / SCHEME  (expect PDF-heavy)", "EN  · PM-KISAN benefits",
     dict(query="pm kisan samman nidhi eligibility and benefits", intent="policy")),
    ("A. POLICY / SCHEME  (expect PDF-heavy)", "HI  · fasal bima yojana",
     dict(query="प्रधानमंत्री फसल बीमा योजना के लाभ और पात्रता क्या है", intent="policy")),

    # B. Field practice / how-to — fusion should push KCC to the top
    ("B. FIELD PRACTICE  (expect KCC-heavy)", "EN  · urea dose in wheat",
     dict(query="how much urea should be applied in wheat at tillering stage",
          intent="field_practice")),
    ("B. FIELD PRACTICE  (expect KCC-heavy)", "HI  · weed control in wheat",
     dict(query="गेहूं में खरपतवार नियंत्रण के लिए कौन सी दवा डालें",
          intent="field_practice")),
    ("B. FIELD PRACTICE  (expect KCC-heavy)", "HING· paddy nursery yellowing",
     dict(query="dhan ki nursery me pili patti ho rahi hai kya karein",
          intent="field_practice")),

    # C. Disease diagnosis — what the vision module hands off to retrieval
    ("C. DISEASE / DIAGNOSIS", "EN  · brown spots on rice",
     dict(query="brown spots on rice leaves what disease is this and how to control")),
    ("C. DISEASE / DIAGNOSIS", "HI  · tomato leaf curl",
     dict(query="टमाटर के पौधे में पत्तियां मुड़ रही हैं क्या करें")),
    ("C. DISEASE / DIAGNOSIS", "HING· wheat yellow rust",
     dict(query="gehu me pila ratua lag gaya hai konsi dawa daale")),

    # D. Metadata-filtered retrieval (M3 §9.5)
    ("D. FILTERED RETRIEVAL", "dosage TABLES only (pdf)",
     dict(query="approved fungicides and dosage for rice blast", only_tables=True)),
    ("D. FILTERED RETRIEVAL", "district canon: allahabad -> prayagraj",
     dict(query="contingency plan for delayed monsoon", district="allahabad")),
    ("D. FILTERED RETRIEVAL", "crop=wheat + season=Rabi (kcc)",
     dict(query="sowing time and seed rate", crop="wheat", season="Rabi")),
    ("D. FILTERED RETRIEVAL", "pinned pdf + scheme_eligibility",
     dict(query="subsidy application process and required documents",
          source_type="pdf", doc_category="scheme_eligibility")),

    # E. Edge cases — abstention, ambiguity, compound intent
    ("E. EDGE CASES", "OFF-DOMAIN en (expect abstain)",
     dict(query="how do I repair my motorcycle engine")),
    ("E. EDGE CASES", "OFF-DOMAIN hi (expect abstain)",
     dict(query="शेयर बाजार में निवेश कैसे करें")),
    ("E. EDGE CASES", "VAGUE (tests §9.7 re-rank case)",
     dict(query="meri fasal kharab ho rahi hai kya karu")),
    ("E. EDGE CASES", "COMPOUND (advice + market)",
     dict(query="wheat me kitna urea dalna hai aur mandi rate kya chal raha hai")),
]

W = 100
print("=" * W)
print("RETRIEVAL INSPECTOR — what each kind of query actually pulls back")
print("=" * W)
if KCC_MAX_CHUNKS:
    print(f"NOTE: DEV subset active (KCC<={KCC_MAX_CHUNKS:,}). Narrow filters (district,")
    print("      season, year) may return few or 0 hits simply because sampling thinned")
    print("      that slice — that is expected here, not a retrieval failure.")
    print("-" * W)

inspection, summary, _cat = [], [], None
for cat, label, kw in INSPECT_QUERIES:
    if cat != _cat:
        _cat = cat
        print(f"\n{'#' * W}\n## {cat}\n{'#' * W}")

    t0 = time.time()
    out = search_agri_knowledge(**{**kw, "top_k": max(TOP_N_SHOW, TOP_K_DEFAULT)})
    ms = (time.time() - t0) * 1000

    params = ", ".join(f"{k}={v!r}" for k, v in kw.items() if k != "query") or "—"
    hits = out.get("results", [])
    n_pdf_h = sum(1 for h in hits[:TOP_N_SHOW] if h["source_type"] == "pdf")
    n_kcc_h = sum(1 for h in hits[:TOP_N_SHOW] if h["source_type"] == "kcc")

    print(f"\n{'-' * W}")
    print(f"[{label}]")
    print(f"  Q      : {kw['query']}")
    print(f"  params : {params}")
    if out.get("tier") == "error":
        print(f"  RESULT : ERROR — {out.get('error', '')[:120]}")
        summary.append((label, "error", 0.0, 0, 0, ms))
        continue
    print(f"  result : tier={out['tier']}  top_raw={out['top_score']:.3f}  "
          f"{ms:.0f}ms  |  top-{TOP_N_SHOW}: {n_pdf_h} pdf / {n_kcc_h} kcc")

    if not hits:
        print("  (no hits — over-restrictive filter, or this slice is absent from the corpus)")
    for i, h in enumerate(hits[:TOP_N_SHOW], 1):
        tbl = " [TABLE]" if h.get("has_table") else ""
        print(f"\n   {i}. raw={h['raw_score']:.3f}  fused={h['fused_score']:.3f}  "
              f"<{h['source_type']}>{tbl}")
        print(f"      {_fmt_citation(h['citation'])}")
        print(f"      \"{_snip(h['text'])}\"")

    summary.append((label, out["tier"], out["top_score"], n_pdf_h, n_kcc_h, ms))
    inspection.append({"category": cat, "label": label, "params": kw,
                       "tier": out["tier"], "top_score": out["top_score"],
                       "latency_ms": round(ms, 1),
                       "hits": [{"raw_score": h["raw_score"], "fused_score": h["fused_score"],
                                 "source_type": h["source_type"], "citation": h["citation"],
                                 "text": h["text"]} for h in hits[:TOP_N_SHOW]]})

# --- Compact routing/scoring overview ---------------------------------------
print(f"\n{'=' * W}")
print("SUMMARY — routing and score distribution at a glance")
print("=" * W)
print(f"  {'query':<42} {'tier':<26} {'top':>6} {'pdf':>4} {'kcc':>4} {'ms':>6}")
print("  " + "-" * (W - 4))
for label, tier, top, npdf, nkcc, ms in summary:
    print(f"  {label:<42} {tier:<26} {top:>6.3f} {npdf:>4} {nkcc:>4} {ms:>6.0f}")

_in_scores = [s for lbl, t, s, *_ in summary if not lbl.startswith("OFF-DOMAIN") and s > 0]
_off_scores = [s for lbl, t, s, *_ in summary if lbl.startswith("OFF-DOMAIN")]
if _in_scores and _off_scores:
    print("  " + "-" * (W - 4))
    print(f"  in-domain min={min(_in_scores):.3f} | off-domain max={max(_off_scores):.3f} "
          f"| separation={min(_in_scores) - max(_off_scores):+.3f}")
    print(f"  current tiers: abstain<{TIER_FALLBACK} <=fallback< {TIER_GROUNDED} <=grounded")

insp_path = os.path.join(RAG_DB_DIR, "retrieval_inspection.json")
with open(insp_path, "w", encoding="utf-8") as f:
    json.dump({"scale": "full" if KCC_MAX_CHUNKS is None else f"dev_{KCC_MAX_CHUNKS}",
               "n_chunks": len(unified), "model": EMBED_MODEL_NAME,
               "queries": inspection}, f, indent=2, ensure_ascii=False)
print(f"\n  full retrieved text saved -> {insp_path}")


RETRIEVAL INSPECTOR — what each kind of query actually pulls back
NOTE: DEV subset active (KCC<=3,000). Narrow filters (district,
      season, year) may return few or 0 hits simply because sampling thinned
      that slice — that is expected here, not a retrieval failure.
----------------------------------------------------------------------------------------------------

####################################################################################################
## A. POLICY / SCHEME  (expect PDF-heavy)
####################################################################################################

----------------------------------------------------------------------------------------------------
[EN  · loan interest subvention]
  Q      : who is eligible for interest subvention on crop loans
  params : intent='policy'
  result : tier=grounded  top_raw=0.999  599ms  |  top-3: 3 pdf / 0 kcc

   1. raw=0.999  fused=1.998  <pdf>
      CHITRAKOOT.pdf | p.1 | contingency_pla